SOLUCIÓN PARCIAL 1, ESTUDIANTE: MARIANA CASTILLO RAMOS

In [57]:
#IMPORTAR LIBRERIAS
import pandas as pd
import numpy as np

In [58]:
#LEER DATOS DE LA GEIH
df_cgsse= pd.read_csv( "https://raw.githubusercontent.com/niconomist98/DataAnalyticsUQ/refs/heads/main/Datos/GEIH2025/cgsse.CSV"
                      ,encoding="latin",sep=";")

df_desocupados= pd.read_csv( "https://raw.githubusercontent.com/niconomist98/DataAnalyticsUQ/refs/heads/main/Datos/GEIH2025/No%20ocupados.CSV"
                      ,encoding="latin",sep=";")    

df_ocupados= pd.read_csv( "https://github.com/niconomist98/DataAnalyticsUQ/raw/refs/heads/main/Datos/GEIH2025/Ocupados.CSV"
                      ,encoding="latin",sep=";")

df_ft= pd.read_csv( "https://github.com/niconomist98/DataAnalyticsUQ/raw/refs/heads/main/Datos/GEIH2025/Fuerza%20de%20trabajo.CSV"
                      ,encoding="latin",sep=";")           

/tmp/ipykernel_5539/2152940998.py:8: DtypeWarning: Columns (0: P6430S1, 1: P6765S1, 2: P6780S1, 3: P6915S1, 4: P7028S1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ocupados= pd.read_csv( "https://github.com/niconomist98/DataAnalyticsUQ/raw/refs/heads/main/Datos/GEIH2025/Ocupados.CSV"
/tmp/ipykernel_5539/2152940998.py:11: DtypeWarning: Columns (0: P3362S7A1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ft= pd.read_csv( "https://github.com/niconomist98/DataAnalyticsUQ/raw/refs/heads/main/Datos/GEIH2025/Fuerza%20de%20trabajo.CSV"


1. Para el total nacional calcule :
 
-población total
-población en edad de trabajar
-población que no está en edad de trabajar
-población económicamente activa
-población económicamente inactiva
-tasa de ocupación
-tasa de desempleo


In [59]:
#CALCULO DE POBLACIÓN TOTAL
ptc = df_cgsse["FEX_C18"].sum()
print('Población total de colombia:', int(ptc))

Población total de colombia: 52045001


In [60]:
#POBLACIÓN EN EDAD DE TRABAJAR Y POBLACIÓN QUE NO ESTA EN EDAD DE TRABAJAR
sin_edad_trabajar = df_cgsse[df_cgsse['P6040']<15]['FEX_C18'].sum()
edad_trabajar = df_cgsse[df_cgsse['P6040']>=15]['FEX_C18'].sum() #PET


print('La poblacion total sin edad para trabajar es:',int(sin_edad_trabajar))
print('La poblacion total con edad para trabjar es:',int(edad_trabajar))

La poblacion total sin edad para trabajar es: 11391585
La poblacion total con edad para trabjar es: 40653416


In [61]:
#Calculado ahora la poblacion economicamente activa y definimos como poblacion activamente economica
#Tenemos que usar el DSI dado que en el modulo de no ocupados contiene informacion de personas que no estan buscando trabajo 
#No ocupados para este analisis son los que estan buscando trabajo activamente DSI=1 
pea = df_ocupados["FEX_C18"].sum() + df_desocupados[df_desocupados['DSI']==1]["FEX_C18"].sum()

#Tambien podriamos usar la variable FT=1 que es la fuerza de trabajo que cumple con las condiciones 
#Tiene trabajo o esta buscando trabajo

#definimos la poblacion economicamente inactiva como
#Usamos la columna FFT que son las personas fuera de la fuerza de trabajo 
pei = df_ft[df_ft['FFT']==1]['FEX_C18'].sum()

print(f'La poblacion economicamente activa es:', int(pea))
print(f'La poblacion economicamente inactiva es:', int(pei))

La poblacion economicamente activa es: 25974846
La poblacion economicamente inactiva es: 14678569


In [62]:
# Ahora vamos a calcular la tasa de ocupacion (TO) con 2 decimales
TO = (df_ocupados['FEX_C18'].sum() / edad_trabajar)

print(f'La tasa de ocupacion: {TO:.2%}')

#Calculando el desempleo (TD) tenemos que

TD = (df_desocupados[df_desocupados['DSI']==1]['FEX_C18'].sum() / pea)
print(f'La tasa de desempleo es {TD:.2%}')

La tasa de ocupacion: 58.43%
La tasa de desempleo es 8.55%


2. Calcule las tasas de desempleo para hombres y mujeres en Risaralda, Caldas , Tolima y Quindío. ¿Cuál es el departamento con mayor y menor tasa de desempleo de los 4 ? 


In [63]:
# 1. Hacemos merge para tener la variable de sexo ('P3271') en el dataset de desocupados
df_desocupados_gen = pd.merge(
    df_desocupados[df_desocupados["DSI"]==1],
    df_cgsse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271']],
    on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'],
    how='left'
)

# 2. Agrupamos por departamento y sexo para contar desocupados usando el factor de expansión
desocupados_sexo = (
    df_desocupados_gen.groupby(['DPTO', 'P3271'])['FEX_C18']
    .sum()
    .reset_index()
    .rename(columns={'FEX_C18': 'desocupados'})
)

# 3. Hacemos lo mismo para el dataset de ocupados
df_ocupados_gen = pd.merge(
    df_ocupados,
    df_cgsse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271']],
    on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'],
    how='left'
)

# 4. Agrupamos ocupados por departamento y sexo
ocupados_sexo = (
    df_ocupados_gen.groupby(['DPTO', 'P3271'])['FEX_C18']
    .sum()
    .reset_index()
    .rename(columns={'FEX_C18': 'ocupados'})
)

# 5. Unimos ocupados y desocupados para calcular la Población Económicamente Activa (PEA)
pea_sexo = pd.merge(ocupados_sexo, desocupados_sexo, on=['DPTO', 'P3271'], how='outer').fillna(0)
pea_sexo['PEA'] = pea_sexo['ocupados'] + pea_sexo['desocupados']

# 6. FILTRO PARA DEPARTAMENTOS ESPECÍFICOS (Eje Cafetero y Tolima)
# Usamos .isin() para seleccionar solo los códigos 63, 66, 17 y 73
departamentos_interes = [63, 66, 17, 73]
pea_sexo = pea_sexo[pea_sexo['DPTO'].isin(departamentos_interes)]

# 7. Calculamos la Tasa de Desempleo (TD)
pea_sexo['TD'] = (pea_sexo['desocupados'] / pea_sexo['PEA']) * 100

# 8. Reemplazamos los códigos de sexo por etiquetas legibles
pea_sexo['Sexo'] = pea_sexo['P3271'].map({1: 'Hombres', 2: 'Mujeres'})

# 9. Creamos un diccionario para ver los nombres de los departamentos en lugar de números
nombres_dpto = {17: 'Caldas', 63: 'Quindío', 66: 'Risaralda', 73: 'Tolima'}
pea_sexo['Departamento'] = pea_sexo['DPTO'].map(nombres_dpto)

# 10. Mostramos la tabla final con las columnas de interés
print(pea_sexo[['Departamento', 'Sexo', 'TD']].sort_values(by='TD', ascending=False))

   Departamento     Sexo         TD
45       Tolima  Mujeres  19.202747
39    Risaralda  Mujeres  10.669927
36      Quindío  Hombres  10.233032
37      Quindío  Mujeres   9.084532
11       Caldas  Mujeres   7.677962
44       Tolima  Hombres   7.220583
38    Risaralda  Hombres   6.074260
10       Caldas  Hombres   5.583183


In [64]:
# 1. Definimos los nombres de los departamentos
nombres_dpto = {17: 'Caldas', 63: 'Quindío', 66: 'Risaralda', 73: 'Tolima'}
pea_sexo['Departamento'] = pea_sexo['DPTO'].map(nombres_dpto)

# 2. Encontramos el índice del departamento con MAYOR y MENOR TD por sexo
# Usamos idxmax() e idxmin() sobre nuestra tabla ya filtrada
td_max = pea_sexo.loc[pea_sexo.groupby("Sexo")["TD"].idxmax()]
td_min = pea_sexo.loc[pea_sexo.groupby("Sexo")["TD"].idxmin()]

# 3. Imprimimos los resultados
print("Departamentos del Eje Cafetero/Tolima con MAYOR tasa de desempleo por sexo:")
print(td_max[['Departamento', 'Sexo', 'TD']])

print("\nDepartamentos del Eje Cafetero/Tolima con MENOR tasa de desempleo por sexo:")
print(td_min[['Departamento', 'Sexo', 'TD']])

# 4. Respuesta final
print(f"\nEl departamento con mayor desempleo en este grupo es {td_max.iloc[0]['Departamento']} "
      f"y el de menor es {td_min.iloc[0]['Departamento']}.")

Departamentos del Eje Cafetero/Tolima con MAYOR tasa de desempleo por sexo:
   Departamento     Sexo         TD
36      Quindío  Hombres  10.233032
45       Tolima  Mujeres  19.202747

Departamentos del Eje Cafetero/Tolima con MENOR tasa de desempleo por sexo:
   Departamento     Sexo        TD
10       Caldas  Hombres  5.583183
11       Caldas  Mujeres  7.677962

El departamento con mayor desempleo en este grupo es Quindío y el de menor es Caldas.


Respuesta punto 2: El Quindio es el departamento con mayor desempleo entre los 4 departamentos analizados y Caldas el que tiene menor tasa de desempleo

3. Tomando en consideración únicamente los siguientes grupos de edad: 

18 a 24
24 a 30
30 a 40
40 a 50 
60 en adelante 

Para Risaralda, Caldas , Tolima y Quindío, cual es el grupo de edad con mayor y menor tasa de desempleo (indicar en cada departamento el grupo de de edad  con mayor y menor tasa de desempleo) ? 


In [65]:
# 1. Definimos los códigos de la región y la función de clasificación
deptos_interes = [17, 63, 66, 73]

def clasificar_edad_trabajo(x):
    if 18 <= x < 24:
        return "18-24"
    elif 24 <= x < 30:
        return "24-30"
    elif 30 <= x < 40:
        return "30-40"
    elif 40 <= x < 50:
        return "40-50"
    elif x >= 60:
        return "60 en adelante"
    else:
        return "Fuera de rango"

# 2. Aplicamos la clasificación al dataset de características generales
df_cgsse["grupo_edad"] = df_cgsse["P6040"].apply(clasificar_edad_trabajo)

# 3. Filtramos y procesamos Desocupados para la región
df_desocupados_reg = df_desocupados[df_desocupados["DPTO"].isin(deptos_interes)].copy()
des_edad = pd.merge(
    df_desocupados_reg[df_desocupados_reg["DSI"]==1],
    df_cgsse[["DIRECTORIO","SECUENCIA_P","ORDEN","grupo_edad"]],
    on=["DIRECTORIO","SECUENCIA_P","ORDEN"],
    how="left"
).groupby(["DPTO","grupo_edad"])["FEX_C18"].sum().reset_index(name="desocupados")

# 4. Filtramos y procesamos Ocupados para la región
df_ocupados_reg = df_ocupados[df_ocupados["DPTO"].isin(deptos_interes)].copy()
ocu_edad = pd.merge(
    df_ocupados_reg,
    df_cgsse[["DIRECTORIO","SECUENCIA_P","ORDEN","grupo_edad"]],
    on=["DIRECTORIO","SECUENCIA_P","ORDEN"],
    how="left"
).groupby(["DPTO","grupo_edad"])["FEX_C18"].sum().reset_index(name="ocupados")

# 5. Calculamos la Tasa de Desempleo (TD) por grupo de edad
pea_edad_reg = pd.merge(ocu_edad, des_edad, on=["DPTO","grupo_edad"], how="outer").fillna(0)
pea_edad_reg["PEA"] = pea_edad_reg["ocupados"] + pea_edad_reg["desocupados"]
pea_edad_reg["TD"] = (pea_edad_reg["desocupados"] / pea_edad_reg["PEA"]) * 100

# 6. Limpiamos: Solo grupos solicitados y ponemos nombres a los departamentos
grupos_validos = ["18-24", "24-30", "30-40", "40-50", "60 en adelante"]
pea_final = pea_edad_reg[pea_edad_reg["grupo_edad"].isin(grupos_validos)].copy()

nombres_depto = {17: 'Caldas', 63: 'Quindío', 66: 'Risaralda', 73: 'Tolima'}
pea_final['Departamento'] = pea_final['DPTO'].map(nombres_depto)

# 7. Obtener el mayor y menor por departamento
mayores_reg = pea_final.loc[pea_final.groupby("Departamento")["TD"].idxmax()]
menores_reg = pea_final.loc[pea_final.groupby("Departamento")["TD"].idxmin()]

print("--- RESULTADOS POR GRUPO DE EDAD ---")
print("\nMAYOR tasa de desempleo por departamento:")
print(mayores_reg[['Departamento', 'grupo_edad', 'TD']])

print("\nMENOR tasa de desempleo por departamento:")
print(menores_reg[['Departamento', 'grupo_edad', 'TD']])

--- RESULTADOS POR GRUPO DE EDAD ---

MAYOR tasa de desempleo por departamento:
   Departamento grupo_edad         TD
0        Caldas      18-24   9.972584
7       Quindío      24-30  14.237924
12    Risaralda      18-24  15.173556
18       Tolima      18-24  19.348020

MENOR tasa de desempleo por departamento:
   Departamento      grupo_edad        TD
4        Caldas  60 en adelante  4.060989
9       Quindío           40-50  6.506359
16    Risaralda  60 en adelante  3.639861
22       Tolima  60 en adelante  9.458632


Respuesta punto 3: En el Quindio, el grupo de edad con mayor tasa de desempleo es el de 24 a 30 años y el grupo con menor TD es el de 40 a 50 años. En Caldas el grupo de edad con mayor TD es el de 18 a 24 años y el grupo con menor TD es el de 60 en adelante. En Risaralda el grupo de edad con mayor TD es de 18 a 24 años igual que en Caldas, y el grupo con menor TD es de 60 en adelante tambien al igual que Caldas. En Tolima tambien ocurre la misma situación que en Caldas y Risaralda

4. Entre Risaralda, Caldas , Tolima y Quindío, cuál es el departamento con menor tasa de desempleo de mujeres entre los 18 y los 25 años ? 


In [66]:
# Se definen los departamentos
deptos_elegidos = [17, 63, 66, 73]

# Preparamos los datos de Desocupados (DSI == 1) incluyendo Sexo y Edad
df_desocupados_reg = pd.merge(
    df_desocupados[(df_desocupados["DSI"] == 1) & (df_desocupados["DPTO"].isin(deptos_elegidos))],
    df_cgsse[["DIRECTORIO", "SECUENCIA_P", "ORDEN", "P3271", "P6040"]], # Agregamos P6040 aquí
    on=["DIRECTORIO", "SECUENCIA_P", "ORDEN"],
    how="left"
)

# Preparamos los datos de Ocupados incluyendo Sexo y Edad
df_ocupados_reg = pd.merge(
    df_ocupados[df_ocupados["DPTO"].isin(deptos_elegidos)],
    df_cgsse[["DIRECTORIO", "SECUENCIA_P", "ORDEN", "P3271", "P6040"]], 
    on=["DIRECTORIO", "SECUENCIA_P", "ORDEN"],
    how="left"
)

# Filtramos para MUJERES (P3271 == 2) entre 18 y 25 años
# Para Desocupadas
desem_joven = df_desocupados_reg[
    (df_desocupados_reg["P3271"] == 2) & 
    (df_desocupados_reg["P6040"].between(18, 25))
].groupby("DPTO")["FEX_C18"].sum()

# Para Ocupadas
ocu_joven = df_ocupados_reg[
    (df_ocupados_reg["P3271"] == 2) & 
    (df_ocupados_reg["P6040"].between(18, 25))
].groupby("DPTO")["FEX_C18"].sum()

# Calculamos la Tasa de Desempleo (PEA = Ocupados + Desocupados)
pea_joven = ocu_joven.add(desem_joven, fill_value=0)
resultado_final = pd.DataFrame({
    "PEA": pea_joven,
    "Desempleadas": desem_joven
}).fillna(0)

resultado_final["Tasa_desempleo(%)"] = (resultado_final["Desempleadas"] / resultado_final["PEA"]) * 100

# Mapear nombres y ordenar
nombres = {17: 'CALDAS', 63: 'QUINDIO', 66: 'RISARALDA', 73: 'TOLIMA'}
resultado_final["Departamento"] = resultado_final.index.map(nombres)
resultado_final = resultado_final.sort_values(by="Tasa_desempleo(%)")

# Mostrar resultado
print("=== Tasa de Desempleo Mujeres 18-25 años (Región) ===")
print(resultado_final[['Departamento', 'Tasa_desempleo(%)']])

=== Tasa de Desempleo Mujeres 18-25 años (Región) ===
     Departamento  Tasa_desempleo(%)
DPTO                                
63        QUINDIO           7.468211
17         CALDAS          14.500102
73         TOLIMA          21.378071
66      RISARALDA          22.992769


Respuesta punto 4: En los grupos de de mujeres entre 18 y 25 años de cada departamento, el departamento que con menor tasa de desempleo de las mujeres de ese rango de edades es el Quindio con un 7.46%

5. ¿Cuántos hombres y mujeres hay en edad de trabajar en el quindio ? ¿Qué porcentaje de la población total representan ? 


In [67]:
# Definimos el código de Quindío y el filtro de edad (PET >= 15 años)
codigo_quindio = 63

# Filtramos el dataset completo 

quindio_total = df_cgsse[df_cgsse['DPTO'] == codigo_quindio].copy()

# Identificamos PET
quindio_pet = quindio_total[quindio_total['P6040'] >= 15].copy()

# Contamos hombres y mujeres en la PET usando el factor de expansión (FEX_C18)
# P3271: 1 = Hombre, 2 = Mujer
conteo_pet = quindio_pet.groupby('P3271')['FEX_C18'].sum()

hombres_pet = conteo_pet.get(1, 0)
mujeres_pet = conteo_pet.get(2, 0)
total_pet_quindio = hombres_pet + mujeres_pet

#Calculamos la población TOTAL del Quindío (PET + No PET) para el porcentaje
poblacion_total_quindio = quindio_total['FEX_C18'].sum()

porcentaje_pet = (total_pet_quindio / poblacion_total_quindio) * 100

#Imprimimos los resultados
print(f"--- POBLACIÓN EN EDAD DE TRABAJAR (PET) EN QUINDÍO ---")
print(f"Hombres en edad de trabajar: {int(hombres_pet):,}".replace(',', '.'))
print(f"Mujeres en edad de trabajar: {int(mujeres_pet):,}".replace(',', '.'))
print(f"Total PET Quindío: {int(total_pet_quindio):,}".replace(',', '.'))

print(f"\n--- ANÁLISIS PORCENTUAL ---")
print(f"La PET representa el {porcentaje_pet:.2f}% de la población total del departamento.")

--- POBLACIÓN EN EDAD DE TRABAJAR (PET) EN QUINDÍO ---
Hombres en edad de trabajar: 247.037
Mujeres en edad de trabajar: 261.387
Total PET Quindío: 508.425

--- ANÁLISIS PORCENTUAL ---
La PET representa el 84.31% de la población total del departamento.


Respuesta punto 5: En el Quindio hay 247.037 hombres en edad de trabajar y 261.387 mujeres en edad de trabajar, el porcentaje total que representan de la población del departamento es el 84.31%

6. ¿Qué porcentaje de la población del Tolima está en edad de trabajar ? 

In [68]:
# Definimos el código del Tolima y filtramos el dataset de personas
codigo_tolima = 73
tolima_total = df_cgsse[df_cgsse['DPTO'] == codigo_tolima].copy()

# Calculamos la población total estimada del departamento usando el factor de expansión
pob_total_tolima = tolima_total['FEX_C18'].sum()

# Filtramos la Población en Edad de Trabajar (PET) que son los de 15 años o más
tolima_pet = tolima_total[tolima_total['P6040'] >= 15]
pob_pet_tolima = tolima_pet['FEX_C18'].sum()

# Realizamos el cálculo del porcentaje final
porcentaje_pet_tolima = (pob_pet_tolima / pob_total_tolima) * 100

# Imprimimos resultados
print(f"Población Total del Tolima: {int(pob_total_tolima):,}".replace(',', '.'))
print(f"Población en Edad de Trabajar (PET): {int(pob_pet_tolima):,}".replace(',', '.'))
print(f"El porcentaje de población en edad de trabajar en Tolima es: {porcentaje_pet_tolima:.2f}%")

Población Total del Tolima: 1.411.965
Población en Edad de Trabajar (PET): 1.087.359
El porcentaje de población en edad de trabajar en Tolima es: 77.01%


Respuesta punto 6: El porcentaje de la población de Tolima que esta en edad de trabajar, es decir, personas mayores de 15 años (PEA+PEI), es de 77.01%

7. ¿ Qué porcentaje de la población de Risaralda es población económicamente inactiva ? 


In [69]:
PEIR= df_ft[(df_ft["FFT"]==1) & (df_desocupados['DPTO']==66)]["FEX_C18"].sum()
PTR= df_cgsse[(df_cgsse['DPTO']==66)]["FEX_C18"].sum()
print(f"Poblacion en edad de trabajar de Risaralda :{PTR}")
print(f"Poblacion economicamente inactiva de Risaralda: {PEIR}")
print(f"Porcentaje de la PEI en Risaralda {(PEIR/PTR)*100}%")

Poblacion en edad de trabajar de Risaralda :1025240.9423573611
Poblacion economicamente inactiva de Risaralda: 207536.6812240729
Porcentaje de la PEI en Risaralda 20.242722724950767%


8. ¿Cuántas personas hay en el Quindío en condición de desempleo entre los 18 y los 30 años ? 


In [70]:
# 1. Unimos desocupados con edad y departamento
df_desocupados_age = pd.merge(
    df_desocupados,
    df_cgsse[['DIRECTORIO','SECUENCIA_P','ORDEN','P6040','DPTO']],
    on=['DIRECTORIO','SECUENCIA_P','ORDEN'],
    how='left'
)

# 2. Filtrar Quindío, desempleados entre 18 y 30 años
desempleo_18_30_quindio = df_desocupados_age[
    (df_desocupados_age['DPTO_y']==63) &
    (df_desocupados_age['DSI']==1) &
    (df_desocupados_age['P6040']>=18) &
    (df_desocupados_age['P6040']<=30)
]['FEX_C18'].sum()

print("Personas desempleadas entre 18 y 30 años en el Quindío:", round(desempleo_18_30_quindio))

Personas desempleadas entre 18 y 30 años en el Quindío: 9480


9. ¿Cuántas personas hay ocupadas en el departamento del Tolima , cuyas edades están entre los 20 y los 25 años , qué porcentaje de la población total del Tolima representan ? 

In [71]:
# 1. Calculamos la población ocupada del Tolima (Código 73)
# IMPORTANTE: Usamos solo df_ocupados en ambos lados del filtro
opt = df_ocupados[(df_ocupados["OCI"]==1) & (df_ocupados['DPTO']==73)]["FEX_C18"].sum()

# 2. Definimos PTT (Población Total del Tolima) 
# Según tus resultados anteriores, este valor es 1.411.965
PTT = df_cgsse[df_cgsse['DPTO'] == 73]['FEX_C18'].sum()

# 3. Ahora imprimimos usando la variable PTT que acabamos de definir
print(f"Poblacion ocupada en el departamento del Tolima: {opt}")
print(f"El porcentaje que representa en la poblacion total del Tolima es: {(opt/PTT)*100:.2f}%")

Poblacion ocupada en el departamento del Tolima: 582285.37132701
El porcentaje que representa en la poblacion total del Tolima es: 41.24%
